# Week 15 - Basic Anomaly Detection

This notebook trains a simple **Isolation Forest** model on heart-rate data to flag irregular values and exposes inference through a small **REST API**.

Use case: wearable heart-rate monitoring where sudden very low or very high readings should be flagged for review.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

RANDOM_STATE = 42

# Create simple heart-rate dataset with mostly normal values + few anomalies
rng = np.random.default_rng(RANDOM_STATE)
normal_hr = rng.normal(loc=78, scale=8, size=280)
anomaly_hr = np.array([35, 42, 155, 168, 172, 180, 34, 160, 170, 39])

heart_rates = np.concatenate([normal_hr, anomaly_hr])
labels = np.concatenate([np.zeros(len(normal_hr), dtype=int), np.ones(len(anomaly_hr), dtype=int)])

df = pd.DataFrame({"heart_rate": heart_rates, "is_anomaly": labels})
df["heart_rate"] = df["heart_rate"].round(1)

X = df[["heart_rate"]]
y = df["is_anomaly"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

model = IsolationForest(
    n_estimators=150,
    contamination=0.04,
    random_state=RANDOM_STATE
)
model.fit(X_train)

# IsolationForest output: -1 means anomaly, 1 means normal
test_pred_raw = model.predict(X_test)
test_pred = np.where(test_pred_raw == -1, 1, 0)

print("Dataset size:", len(df))
print("Anomaly count:", int(df["is_anomaly"].sum()))
print("\nClassification report on holdout split:")
print(classification_report(y_test, test_pred, digits=4))

df.head()

Dataset size: 290
Anomaly count: 10

Classification report on holdout split:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        56
           1     1.0000    1.0000    1.0000         2

    accuracy                         1.0000        58
   macro avg     1.0000    1.0000    1.0000        58
weighted avg     1.0000    1.0000    1.0000        58



,heart_rate,is_anomaly
0,80.4,0
1,69.7,0
2,84.0,0
3,85.5,0
4,62.4,0


In [4]:
def detect_heart_rate_anomaly(heart_rate: float) -> dict:
    value = pd.DataFrame({"heart_rate": [float(heart_rate)]})
    raw_pred = model.predict(value)[0]
    score = float(model.decision_function(value)[0])

    return {
        "heartRate": float(heart_rate),
        "anomaly": bool(raw_pred == -1),
        "label": "anomaly" if raw_pred == -1 else "normal",
        "anomalyScore": round(score, 6)
    }

# Quick sanity check
sample_results = [
    detect_heart_rate_anomaly(72),
    detect_heart_rate_anomaly(88),
    detect_heart_rate_anomaly(170)
]

sample_results

[{'heartRate': 72.0,
  'anomaly': False,
  'label': 'normal',
  'anomalyScore': 0.243822},
 {'heartRate': 88.0,
  'anomaly': False,
  'label': 'normal',
  'anomalyScore': 0.18687},
 {'heartRate': 170.0,
  'anomaly': True,
  'label': 'anomaly',
  'anomalyScore': -0.094343}]

In [3]:
from flask import Flask, jsonify, request

app = Flask(__name__)

@app.get("/api/health")
def health_check():
    return jsonify({"status": "ok", "service": "week15-basic-anomaly-detection"})

@app.post("/api/anomaly/detect")
def detect_single():
    body = request.get_json(silent=True) or {}
    if "heartRate" not in body:
        return jsonify({"message": "heartRate is required"}), 400

    try:
        heart_rate = float(body["heartRate"])
    except (TypeError, ValueError):
        return jsonify({"message": "heartRate must be a number"}), 400

    result = detect_heart_rate_anomaly(heart_rate)
    return jsonify(result), 200

@app.post("/api/anomaly/batch")
def detect_batch():
    body = request.get_json(silent=True) or {}
    heart_rates = body.get("heartRates")

    if not isinstance(heart_rates, list) or not heart_rates:
        return jsonify({"message": "heartRates must be a non-empty array"}), 400

    try:
        results = [detect_heart_rate_anomaly(float(v)) for v in heart_rates]
    except (TypeError, ValueError):
        return jsonify({"message": "heartRates must contain only numeric values"}), 400

    return jsonify({"count": len(results), "results": results}), 200

In [5]:
# Quick API check without starting external server
with app.test_client() as client:
    health_resp = client.get("/api/health")
    detect_resp = client.post("/api/anomaly/detect", json={"heartRate": 172})

print("Health:", health_resp.status_code, health_resp.get_json())
print("Detect:", detect_resp.status_code, detect_resp.get_json())

Health: 200 {'service': 'week15-basic-anomaly-detection', 'status': 'ok'}
Detect: 200 {'anomaly': True, 'anomalyScore': -0.096761, 'heartRate': 172.0, 'label': 'anomaly'}


## REST API (Flask)

This API exposes the trained in-memory model:

- `GET /api/health`
- `POST /api/anomaly/detect` with JSON payload `{ "heartRate": 95 }`
- `POST /api/anomaly/batch` with JSON payload `{ "heartRates": [72, 89, 170] }`

In [ ]:
# Run API from notebook (stop execution of this cell to stop server)
app.run(host="127.0.0.1", port=5060, debug=False, use_reloader=False)